In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "total_accuracy",
    "theme_accuracy",
    "theme_precision",
    "theme_recall",
    "theme_f1",
    "topic_accuracy",
    "topic_precision",
    "topic_recall",
    "topic_f1",
    "concept_accuracy",
    "concept_precision", 
    "concept_recall",
    "concept_f1",
]

def score_table(jobs, cols, measure_hallucination_detection=True, mark_gen_errors_as_hallucinations=False):

    print(f"Processing {len(jobs)} jobs.")    
    table = {col_name: [] for col_name in cols}
    num_parse_errors = 0
    num_gen_errors = 0
    num_rows = 0
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        num_rows += len(df)

        # LLM failed to produce proper json
        mask_parse_errors = ~(df[PRED_COLS] != '"PARSE ERROR"').all(axis=1)
        num_parse_errors += list(mask_parse_errors).count(True)

        # LLM did not follow instructions, also includes parse errors
        mask_gen_errors = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        num_gen_errors += list(mask_gen_errors).count(True) - list(mask_parse_errors).count(True)

        if mark_gen_errors_as_hallucinations:
            # Set labels
            df.loc[mask_gen_errors, ['themeCorrect', 'topicCorrect', 'usesAdditionalConcepts']] = ['"no"', '"no"', '"yes"']
        else: 
            # Remove erroneous rows
            df = df[~mask_gen_errors]

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df, measure_hallucination_detection=measure_hallucination_detection).items():
            table[name].append(score)

    print(f"Processed {num_rows} rows.")
    print(f"Number of rows with JSON parse errors: {num_parse_errors}.")
    print(f"Number of rows where LLM failed to adhere to response schema: {num_gen_errors}.")
    print(f"Total number of erroneous rows: {num_parse_errors + num_gen_errors}.")
    print(f"Error rate: {(num_parse_errors + num_gen_errors) / num_rows}")

    return table

In [4]:
def aggregate_results(df, by_cols, cols, funs=None):
    grouped = df.groupby(
        by=by_cols,
        as_index=True,
    )
    
    if funs is None:
        funs = [
        "mean",
        "std",
        "count"
    ]

    agg = grouped.agg({col: funs for col in cols})

    return agg

In [5]:
def bold_extreme_values(s):
    # Bold max for mean
    if "mean" in s.name:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    if "std" in s.name:
        is_min = s == s.min()
        return ['font-weight: bold' if v else '' for v in is_min]

    return ['' for v in s]

In [6]:
# Collect raw data
basepath = "./outputs/v2"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res1 = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=False))
res1 = prettify_table(res1)

Processing 330 jobs.
Processed 176550 rows.
Number of rows with JSON parse errors: 4005.
Number of rows where LLM failed to adhere to response schema: 38.
Total number of erroneous rows: 4043.
Error rate: 0.02290002832058907


In [7]:
res2 = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=True))
res2 = prettify_table(res2)

Processing 330 jobs.
Processed 176550 rows.
Number of rows with JSON parse errors: 4005.
Number of rows where LLM failed to adhere to response schema: 38.
Total number of erroneous rows: 4043.
Error rate: 0.02290002832058907


In [8]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format
idx = pd.IndexSlice

agg1 = aggregate_results(res1, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])
# Drop count
#agg1 = agg1.loc[idx['Llama-3.1-8B-Instruct', 1, "positive", :], (slice(None), ["mean"])]
# Highlight max values for each column
agg1.style.apply(bold_extreme_values, axis=0)

In [9]:
agg2 = aggregate_results(res2, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])
# Drop count
agg2 = agg2.loc[idx['Llama-3.1-8B-Instruct', 1, "positive", :], (slice(None), ["mean"])]
# Highlight max values for each column
agg2.style.apply(bold_extreme_values, axis=0)